### This ARTICLES notebook 

- Loads journal data into the database  
- Finds the ISSN of Domingo's Incites journals  
- Examines the "completeness" of the Incites journals

- Extracts the works for each journal into the cache

- Load the works into the database after  
    - Filters works into a flat table (work_id, doi, source, host, citation_count etc)  
    - Flattens the authorships table for each work (author_id, institution_id etc)  
    - Filters the reference list to make the cited table (When inverted these are the endogenous citations)  

Note that at this stage the works have been filtered by publicaiotn date and type (articles, etc)


In [10]:
%run aaa_setup.ipynb
# import duckdb
# import pandas as pd
# from pathlib import Path
# import diskcache
# from itertools import chain

# from utils.pandas_setup import pandas_setup
# pandas_setup()

# import pyalex
# from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
# pyalex.config.email = "Lawrence.Cram@anu.edu.au"
# pyalex.config.max_retries = 8
# pyalex.config.retry_backoff_factor = 0.1
# pyalex.config.retry_http_codes = [429, 500, 503]

# import contextlib
# from unidecode import unidecode
# from nameparser import HumanName

# MY_DATA_PATH = Path('../DATA/')
# MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
# MY_CACHE_FILE = Path('/home/lc/m/.cache/economicsbusiness/cache.db')
# DATAFILES_PATH = Path('../DATAFILES')


# def normalise_name(in_name: str=None) -> list:
#     # print(f'{in_name = }')
#     in_name = ' '.join([part.strip() for part in unidecode(in_name).split(' ')])
#     in_name = in_name.title()
#     name = HumanName(in_name)
#     if name.middle == "":
#         fullname = f'{name.first} {name.last}'
#     else:
#         fullname = f'{name.first} {name.middle} {name.last}'
#     # return {'first': name.first, 'middle': name.middle, 'family': name.last, 'fullname': fullname}
#     return [name.first, name.middle, name.last, fullname]

In [11]:
# class SetUp:

#     def __init__(self):
#         self._setup_db()
#         self._setup_cache()
#         return
    
#     def _setup_db(self):
#         self.db = duckdb.connect()
#         self.db.sql(f"ATTACH IF NOT EXISTS '{str(MY_DATABASE_FILE)}' AS project")
#         self.db.sql(""" SET memory_limit = '56GB';
#                         SET threads = 6;
#                         SET preserve_insertion_order = false;
#                         SET order_by_non_integer_literal=true;
#                         SET enable_progress_bar = true;
#                         SET temp_directory = '/home/lc/m/.tmp';
#                     """)
#         for tab in ['project.edge_list_sources', 'project.edge_list_institutions', 'project.edge_list_both']: 
#             self.db.sql(f"DROP TABLE IF EXISTS {tab}")

#         # with contextlib.suppress(Exception):
#         #     self.db.create_function('normalise_name', 
#         #                                 normalise_name, 
#         #                                 return_type=duckdb.duckdb.typing.DuckDBPyType(dict[str, str]), 
#         #                                 exception_handling='return_null',
#         #                                 null_handling='special',
#         #                                 side_effects=True
#         #                             )
                        
#         self.db.sql("SHOW ALL TABLES").show()
#         return
    
#     def _setup_cache(self):
#         self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
#         print(f'{self.cache.check() = }')
#         print(f'{self.cache.volume() = }')
#         return
    
#     def name_of_global_obj(self, obj=None):
#         for objname, oid in globals().items():
#             if oid is obj:
#                 return objname

In [12]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_journals(self):
        # Extract inCites-OpenAlex journal table (ISSN and OA journal_id)
        self.journals = self.db.sql("SELECT * FROM project.sources_oa_incites").df().\
            rename(columns={'id': 'source_id'}).sort_values('works_count').reset_index()
        print(f'{self.journals.shape = }\n{self.journals.head()}')
        return
    
    def extract_works_by_journal(self):
        # Extract OA works for the journal set, for publication years 2010+ to now
        hold = []
        for row in self.journals.itertuples():
            source_id = row.source_id
            reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
            if oa := self._read_openalex(reader=reader):
                # print(f'EXTRACTED {len(oa) = } WORKS FOR {row.display_name = }')
                hold.extend(oa)
            else:
                print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
            if row.Index % 25 == 0:
                print(f'{row.Index}/{len(self.journals)} completed')
        self._load_articles(hold=hold)                
        return
    
    def _read_openalex(self, reader=None):
        # using the pyalex API call "reader" as key, run the API call, cache the response, and return it (JSON) 
        if result := self.cache.get(reader):
            # print('result from cache')
            return result
        try:
            result = list(chain(*eval(reader).paginate(per_page=200)))
        except Exception as e:
            print(f'FAILED TO READ cache or OpenAlex API with {reader = }')
            print(f'{e = }')
            return
        # print('result from web')
        self.cache[reader] = result
        return result
    
    def _load_articles(self, hold=None):
        print('_load articles')
        # convert the downloaded OA works JSON to a dataframe and split out authorships and reference list
        df = pd.DataFrame.from_records(hold)
        if 'is_authors_truncated' not in df.columns:
            df['is_authors_truncated'] = pd.NA
        df = df.rename(columns={'id': 'work_id'}).set_index('work_id')
        mask = [t in ['article', 'review', 'preprint', 'letter'] and w > 0 for t, w in zip(df['type'], df.referenced_works_count)]
        df = df[mask]
        print(f'{df.shape = }\n{df.head()}')
        self._load_basic_table(df=df)
        self._load_authorships(df=df)
        self._load_referenced_works(df=df)
        return

    def _load_basic_table(self, df=None):
        # load the database with the flatened and editted works table
        cols = ["doi", "title", "display_name", "publication_year", "primary_location", "type",
                "countries_distinct_count", "institutions_distinct_count", "fwci", "has_fulltext", 
                "cited_by_count", "biblio", "is_retracted", "is_paratext", 
                "referenced_works_count", "cited_by_api_url", "updated_date", "created_date", "is_authors_truncated"]
        df1 = df[cols]
        df_biblio = pd.DataFrame(df1['biblio'].values.tolist(), index=df1.index)
        print(f'{df_biblio.shape = }\n{df_biblio.head()}\n{df_biblio.head()}')

        df2 = pd.DataFrame(df1['primary_location'].values.tolist(), index=df1.index)[['source']]
        df_source = pd.DataFrame(df2['source'].values.tolist(), index=df2.index).\
            rename(columns={'id': 'source_id', 'display_name': 'source_name', 'host_organization': 'host_id', 'host_organization_name': 'host_name'})\
                [['source_id', 'source_name', 'host_id', 'host_name']]
        print(f'{df_source.shape = }\n{df_source.head()}\n{df_source.info()}')
        df3 = pd.concat([df1, df_biblio, df_source], axis=1).drop(columns=['biblio', 'primary_location']).reset_index()
        print(f'{df3.shape = }\n{df3.head()}\n{df3.info()}')
        self.db.sql("CREATE OR REPLACE TABLE project.works_full AS (SELECT * FROM df3)")
        self.db.sql("SELECT * FROM project.works_full").show()
        self.db.sql("SELECT count(*) FROM project.works_full").show()
        return

    def _load_authorships(self, df=None):
        # load database with authorships table after flattening the authors and institutions
        cols = ["authorships"]
        df1 = df[cols]
        df2 = df1.explode('authorships').dropna()
        df3 = pd.DataFrame(df2['authorships'].values.tolist(), index=df2.index)
        df4 = pd.DataFrame(df3['author'].values.tolist(), index=df2.index).rename(columns={'id': 'author_id', 'display_name': 'author_name'})
        df4['author_name'] = [normalise_name(n)[-1] for n in df4.author_name]
        df4['author_name'] = ['Georg Weizsaecker' if n == 'Georg Weizsacker' else n for n in df4.author_name]
        df4['author_name'] = ['Marta Reynal' if n == 'Marta Reynal-Querol' else n for n in df4.author_name]
        df4['author_name'] = ['Nicola Fuchs-Schuendeln' if n == 'Nicola Fuchs-Schundeln' else n for n in df4.author_name]        
        df5 = pd.concat([df4, df3], axis=1).set_index(['author_id', 'author_name', 'orcid'], append=True)
        df5 = df5.drop(columns=['author_position', 'author', 'countries', 'is_corresponding', 'raw_author_name', 'raw_affiliation_strings', 'affiliations'])   
        df6 = df5.explode('institutions').dropna()
        df7 = pd.DataFrame(df6['institutions'].values.tolist(), index=df6.index).\
            rename(columns={'id': 'institution_id', 'display_name': 'institution_name'}).drop(columns=['lineage'])
        df7 = df7.reset_index()
        self.db.sql("CREATE OR REPLACE TABLE project.authorships_full AS (SELECT * FROM df7)")
        self.db.sql("SELECT * FROM project.authorships_full").show()
        self.db.sql("SELECT count(*) FROM project.authorships_full").show()
        return

    def _load_referenced_works(self, df=None):
        # load database with referenced works table
        cols = ["referenced_works"]
        ddf = df[cols].reset_index()
        print(ddf.head())
        self.db.sql("CREATE OR REPLACE TABLE project.cited_full AS SELECT * FROM ddf")
        self.db.sql("SELECT * FROM project.cited_full").show()
        self.db.sql("SELECT count(*) FROM project.cited_full").show()
        return


In [13]:
class ExtractAuthors(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def build_author_list(self):
        df = self.db.sql("SELECT DISTINCT author_id FROM project.authorships WHERE author_id NOT NULL ORDER BY author_id").df()
        self.author_ids = df.author_id.str.replace('https://openalex.org/', '').tolist()
        print(f'{len(self.author_ids) = } {self.author_ids[:4] = }')
        return

    def extract_authors(self):
        # Extract OA authors for the journal set
        hold = []
        block_length = 100
        start = 0
        block_total = len(self.author_ids)//block_length + 1
        print(f'extract authors {start = } {block_length = } {block_total = }')
        for block_count in range(block_total):
            authors = '|'.join(self.author_ids[start: start+block_length])
            start = start + block_length
            reader = rf'Authors().filter(id="{authors}")'
            if oa := self._read_openalex(reader=reader):
                # print(f'EXTRACTED {len(oa) = } authors FOR {authors = }')
                hold.extend(oa)
            else:
                print(f'OpenAlex does not have authors for {authors = }')
            if block_count % 1 == 0:
                print(f'{block_count = } {block_count*block_length}/{len(self.author_ids)} completed')
        self._load_authors(hold=hold)
        return
    
    def _read_openalex(self, reader=None):
        # using the pyalex API call "reader" as key, run the API call, cache the response, and return it (JSON) 
        if result := self.cache.get(reader):
            return result
        try:
            result = list(chain(*eval(reader).paginate(per_page=200)))
        except Exception as e:
            print(f'FAILED TO READ cache or OpenAlex API with {reader = }')
            print(f'{e = }')
            return
        self.cache[reader] = result
        return result
    
    def _load_authors(self, hold=None):
        df = pd.DataFrame(hold).rename(columns={'id': 'author_id', 'display_name': 'author_name'})
        for col in ['2yr_mean_citedness', 'h_index', 'i10_index']:
            df[col] = [s.get(col) for s in df.summary_stats]
        cols = ['author_id', 'orcid', 'author_name', 'display_name_alternatives', 'works_count', 'cited_by_count', '2yr_mean_citedness', 'h_index', 'i10_index']
        df = df[cols]
        df[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in df.author_name]
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE project.authors_full AS SELECT * FROM df")
        self.db.sql("SELECT * FROM project.authors_full").show()
        self.db.sql("SELECT count(*) FROM project.authors_full").show()
        return        

In [ ]:
    
class MatchDomingoSample(SetUp):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../RESULTS/researchers_results.xlsx').drop(columns=['Unnamed: 0', 'NAME'])
        print(f'{sample.shape = }\n{sample.head()}')
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        self.db.sql("CREATE OR REPLACE TABLE econ.domingo_sample AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        return

    
    def match_sample(self):
        print('_match_sample')
        sample = self.db.sql("SELECT * FROM econ.domingo_sample").df()
        print(f'{sample.shape = }\n{sample.head()}')
        authors = self.db.sql("SELECT * FROM econ.authors").df().sort_values(['cited_by_count', 'works_count'], ascending=[False, False])\
            .reset_index()[['author_id', 'author_name', 'orcid', 'display_name_alternatives', 'cited_by_count', 'works_count']]
        # authors = authors[authors.cited_by_count>100].dropna()
        for row in authors.itertuples():
            authors.at[row.Index, 'display_name_alternatives'] = [normalise_name(n)[-1] if isinstance(n, str) else n for n in row.display_name_alternatives]
        print(f'{authors.shape = }\n{authors.head()}')

        for row in sample.itertuples():
            for row1 in authors.itertuples():
                if row.fullname in row1.display_name_alternatives:
                    sample.at[row.Index, 'author_id'] = row1.author_id
                    sample.at[row.Index, 'orcid'] = row1.orcid
                    break
        print(f'{sample.shape = }\n{sample.head(24)}')
        print(f'{sample[sample.author_id.isna()].shape = }\n{sample[sample.author_id.isna()].head(24)}')
        self.db.sql("CREATE OR REPLACE TABLE memory.matched AS SELECT * FROM sample")
        return

    def load_sample(self):
        print('compare sample')
        sql = """
            CREATE OR REPLACE TABLE econ.sample_matched AS
                SELECT Research_Profile,
                        m.orcid,
                        m.author_id,
                        m.first,
                        m.last,
                        m.fullname,
                        ACR,
                        PUB,
                        CIT,
                        HCP,
                        suma,          
                        coc,      
                        score, 
                        "Group",
                        works_count_endogenous, 
                        citations_endogenous,
                        hca_endogenous,	                    	
                        works_count_total,
                        cited_by_count,
                        hca_total,
                        "2yr_mean_citedness",
                        h_index,
                        citations_total_ AS citations_total_oa,	
                    FROM memory.matched m 
                    LEFT JOIN citation_summary s
                        ON s.author_id = m.author_id
                    ORDER BY hca_endogenous DESC, citations_endogenous DESC
            """
        self.db.sql(sql)
        sample_align = self.db.sql("SELECT * FROM econ.sample_matched").df()
        print(f'{sample_align.shape = }\n{sample_align.head()}')
        with pd.ExcelWriter('../DATA/domingo_sample_match.xlsx') as writer:
            sample_align.to_excel(writer, index=False, sheet_name='full_match')
            df = sample_align.groupby('Research_Profile').first().reset_index()
            df.to_excel(writer, index=False, sheet_name='filtered')
            print(f'{df.shape = }\n{df.head()}')
            df[df.author_id.isna()].to_excel(writer, index=False, sheet_name='filtered_unmatched')
            print(f'{df[df.author_id.isna()].shape = }\n{df[df.author_id.isna()].head()}')
        return

In [15]:
def main():

    jetl = ArticlesETL()
    jetl.extract_journals()
    jetl.extract_works_by_journal()

    ea = ExtractAuthors()
    ea.build_author_list()
    ea.extract_authors()

    mds = MatchDomingoSample()

In [16]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────────┬───────────┐
│ database │ schema  │         name         │     column_names     │             column_types              │ temporary │
│ varchar  │ varchar │       varchar        │      varchar[]       │               varchar[]               │  boolean  │
├──────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────────┼───────────┤
│ memory   │ main    │ t1                   │ [i, j]               │ [INTEGER, INTEGER]                    │ false     │
│ project  │ main    │ author_works_counts  │ [author_id, author…  │ [VARCHAR, VARCHAR, BIGINT, BIGINT, …  │ false     │
│ project  │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorships          │ [work_id, author_i…  │ [VARCHAR, VARCHAR, VARCHAR, VARCHAR…  │ false     │
│ project  │ main    │ authorshi

NameError: name 'normalise_name' is not defined